# Семинар 2. Свой репозиторий и Python как язык

Первая пара сегодня была про данные. Эта — про инструмент, которым вы будете их разбирать до декабря.

Работа идёт в двух местах:

1. **В терминале** — свой репозиторий, ветка, pull request и намеренно сломанное слияние. Это отдельный файл, откройте его рядом: **[С2-git-практикум.md](С2-git-практикум.md)**.
2. **Здесь, в ноутбуке** — Python как язык.

Второй пункт важнее, чем кажется на слух. pandas начинается со следующей лекции, и он спрячет от вас циклы, словари и множества. Спрячет — не значит отменит: когда расчёт разойдётся с ожиданием, разбираться вы будете именно на этом уровне.

> **Как работать.** Ячейки запускаются `Shift+Enter`, сверху вниз. Базу сегодня не трогаем: все данные лежат в первой ячейке, поэтому ноутбук работает и без сети.

---

## 1. Как выглядят данные до pandas

In [1]:
# Четырнадцать платежей ПРАЙМ. Ровно то, что вернул бы вчерашний запрос.
PAYMENTS = [
    {"payment_id": "p01", "client": "c17", "tariff": "Базовый", "channel": "organic", "amount": 199.0, "status": "success"},
    {"payment_id": "p02", "client": "c17", "tariff": "Базовый", "channel": "organic", "amount": 199.0, "status": "success"},
    {"payment_id": "p03", "client": "c04", "tariff": "Расширенный", "channel": "performance", "amount": 399.0, "status": "success"},
    {"payment_id": "p04", "client": "c04", "tariff": "Расширенный", "channel": "performance", "amount": 399.0, "status": "failed"},
    {"payment_id": "p05", "client": "c04", "tariff": "Расширенный", "channel": "performance", "amount": 399.0, "status": "success"},
    {"payment_id": "p06", "client": "c23", "tariff": "Базовый", "channel": "social", "amount": 199.0, "status": "success"},
    {"payment_id": "p07", "client": "c23", "tariff": "Базовый", "channel": "social", "amount": 199.0, "status": "refunded"},
    {"payment_id": "p08", "client": "c11", "tariff": "Расширенный", "channel": "partner_telecom", "amount": 399.0, "status": "success"},
    {"payment_id": "p09", "client": "c11", "tariff": "Расширенный", "channel": "partner_telecom", "amount": 399.0, "status": "success"},
    {"payment_id": "p10", "client": "c11", "tariff": "Расширенный", "channel": "partner_telecom", "amount": 399.0, "status": "success"},
    {"payment_id": "p11", "client": "c31", "tariff": "Базовый", "channel": "organic", "amount": 199.0, "status": "failed"},
    {"payment_id": "p12", "client": "c05", "tariff": "Расширенный", "channel": "crm", "amount": 399.0, "status": "success"},
    {"payment_id": "p13", "client": "c05", "tariff": "Расширенный", "channel": "crm", "amount": 399.0, "status": "failed"},
    {"payment_id": "p14", "client": "c17", "tariff": "Базовый", "channel": "organic", "amount": 199.0, "status": "refunded"},
]

len(PAYMENTS)

14

Это **список словарей** — ровно то, что отдаёт база через `cursor.fetchall()` и что придёт из внешнего API в теме 10. `DataFrame` появится только на следующей лекции; данные в таком виде вы будете встречать весь курс.

Четырнадцать строк вместо двадцати миллионов — чтобы каждое число можно было пересчитать в уме и поймать себя на ошибке.

In [ ]:
PAYMENTS[0]

Один платёж — один словарь. Ключи одинаковые у всех строк, и это единственная причина, по которой из такого списка потом получится таблица.

> **Квадратные скобки у словаря — не то же самое, что у списка.** У списка в них номер, у словаря — ключ. `PAYMENTS[0]` это «нулевая строка», `PAYMENTS[0]["amount"]` — «колонка `amount` в нулевой строке».

---

## 2. Сколько мы заработали

In [ ]:
total = 0.0
for p in PAYMENTS:
    total += p["amount"]

total

Четыре тысячи триста восемьдесят шесть рублей.

И это неправда.

In [ ]:
from collections import Counter

Counter(p["status"] for p in PAYMENTS)

`success` — деньги пришли. `failed` — попытка списания не прошла, банк отказал. `refunded` — деньги вернули клиенту.

Выручка — только первое. Правило с прошлой пары никуда не делось: **сначала вопрос, потом расчёт.** Вопрос звучал «сколько мы заработали», а не «сколько раз мы попытались списать».

In [ ]:
revenue = 0.0
for p in PAYMENTS:
    if p["status"] == "success":
        revenue += p["amount"]

revenue

**2991 вместо 4386. Разница — 1395 рублей, почти треть.**

Ошибка ровно того же сорта, что вчерашний двойной счёт: код отработал без единого предупреждения и выдал правдоподобное число.

In [ ]:
sum(p["amount"] for p in PAYMENTS if p["status"] == "success")

Одна строка вместо четырёх, и результат тот же. Читается справа налево от `for`:

| Кусок | Что делает |
|---|---|
| `for p in PAYMENTS` | пройти по всем платежам |
| `if p["status"] == "success"` | оставить только успешные |
| `p["amount"]` | взять из каждого сумму |
| `sum(...)` | сложить |

Это **генераторное выражение**. Если обернуть его в квадратные скобки — получится **генератор списка** (list comprehension), одна из двух-трёх конструкций, на которых стоит весь код курса.

In [ ]:
import sys

as_list = [p["amount"] for p in PAYMENTS if p["status"] == "success"]  # квадратные — список
as_gen = (p["amount"] for p in PAYMENTS if p["status"] == "success")   # круглые — генератор

print("на девяти платежах")
print("  список   :", sys.getsizeof(as_list), "байт")
print("  генератор:", sys.getsizeof(as_gen), "байт")

print("на миллионе чисел")
print("  список   :", sys.getsizeof([x for x in range(1_000_000)]), "байт")
print("  генератор:", sys.getsizeof((x for x in range(1_000_000))), "байт")

Посмотрите на первую пару чисел внимательно: **на девяти платежах генератор занимает больше, чем список.** Сам объект-генератор весит две сотни байт, и на такой мелочи это вся его масса.

А теперь на вторую пару. Список из миллиона чисел — восемь с половиной мегабайт. Генератор — по-прежнему меньше двухсот байт, потому что он не построил ничего: он умеет выдавать следующее число по запросу и не хранит остальные.

> **Мерить надо на том объёме, на котором будете работать.** На четырнадцати строках любая оптимизация выглядит бессмысленной, и это ничего не говорит о двадцати миллионах.

Отсюда практическое правило: если результат нужен один раз — круглые скобки. Если ходить по нему придётся дважды — квадратные, и вот почему.

In [ ]:
print("сумма по списку      :", sum(as_list))
print("сумма по списку опять:", sum(as_list))
print("сумма по генератору  :", sum(as_gen))
print("а теперь ещё раз     :", sum(as_gen))

**Последняя строка — ноль.** Генератор одноразовый: он выдал все девять значений и закончился, а `sum` по пустому генератору честно вернул `0`.

Это не ошибка и не падение. Просто в отчёте появится ноль, и никто вам об этом не скажет — ещё один случай того же класса, что и потерянная треть выручки.

---

## 3. Распаковка: как читать чужой код

In [ ]:
for i, p in enumerate(PAYMENTS[:3], start=1):
    print(i, p["client"], p["amount"])

`enumerate` выдаёт пары «номер, элемент», а запись `for i, p in ...` их **распаковывает** — сразу в две переменные. Без неё пришлось бы заводить счётчик руками и не забывать его увеличивать.

`start=1` — потому что людям показывают нумерацию с единицы, а Python считает с нуля.

In [ ]:
clients = ["c17", "c04", "c11"]
amounts = [398.0, 798.0, 1197.0]

for client, amount in zip(clients, amounts):
    print(client, "->", amount)

`zip` склеивает два списка в пары. Если списки разной длины, он молча останавливается на коротком — запомните это, здесь теряются строки.

In [ ]:
first, *rest = [199.0, 399.0, 199.0, 399.0]

print("первый  :", first)
print("остальные:", rest)

Звёздочка слева от переменной означает «а сюда сложи всё остальное». Пригодится, когда из строки данных нужен первый элемент отдельно, а хвост — целиком.

---

## 4. Группировка без pandas

In [ ]:
from collections import defaultdict

by_tariff = defaultdict(float)
for p in PAYMENTS:
    if p["status"] == "success":
        by_tariff[p["tariff"]] += p["amount"]

dict(by_tariff)

`defaultdict(float)` — словарь, который на незнакомый ключ не падает с `KeyError`, а сам заводит `0.0`. Без него первая же строка потребовала бы проверки «а есть ли уже такой тариф».

> **На следующей лекции это будет одна строка:** `df.groupby("tariff")["amount"].sum()`. Сегодня важно, что вы знаете, что у неё внутри — потому что когда `groupby` даст неожиданное число, объяснять его придётся вот на этом уровне.

In [ ]:
Counter(p["channel"] for p in PAYMENTS if p["status"] == "success")

`Counter` — тот же приём для случая «просто посчитать, сколько раз встретилось». Обратите внимание: он считает **строки**, а не рубли. Число платежей и сумма платежей — разные вопросы, и путать их будут весь семестр.

---

## 5. Топ-3 канала

In [ ]:
by_channel = defaultdict(float)
for p in PAYMENTS:
    if p["status"] == "success":
        by_channel[p["channel"]] += p["amount"]

top = sorted(by_channel.items(), key=lambda kv: kv[1], reverse=True)
top[:3]

`sorted` по умолчанию сортирует по самому элементу. У словаря элемент — пара `("organic", 398.0)`, и сортировка по ней пошла бы по алфавиту. Аргумент `key` говорит, **по какой части** сортировать: `kv[1]` — это сумма.

`lambda` здесь — просто способ написать однострочную функцию прямо на месте, не придумывая ей имени.

---

## 6. `b = a` не делает копию

In [ ]:
tariffs = ["Базовый", "Расширенный"]
backup = tariffs

tariffs.append("Промо")

print("tariffs:", tariffs)
print("backup :", backup)

**В `backup` три элемента, хотя его никто не трогал.**

Причина в том, что `backup = tariffs` не создало второй список. Оно дало **второе имя тому же самому списку**. Список в памяти один, имён на него два, и изменение через любое из имён видно через оба.

Эта задача была во входной диагностике, и она разделяет группу лучше всех остальных: она отличает тех, кто писал код, от тех, кто про него читал.

In [ ]:
tariffs = ["Базовый", "Расширенный"]
backup = list(tariffs)   # вот теперь настоящая копия

tariffs.append("Промо")

print("tariffs:", tariffs)
print("backup :", backup)

> **Зачем это в курсе про данные.** В pandas 3 действует Copy-on-Write, и там та же история поворачивается другой стороной: срез таблицы больше не даёт писать в исходный кадр — запись просто молча не доходит. Учебники, написанные до pandas 3.0, в этом месте вводят в заблуждение. Разбираем на теме 03.

---

## 7. Функция, которая переживёт грязные данные

In [ ]:
# Так выглядит выгрузка, которую прислали из соседнего отдела.
RAW = [
    {"client": "c17", "amount": "199.00"},
    {"client": "c04", "amount": "399,00"},   # запятая вместо точки
    {"client": "c23", "amount": "1 200"},    # пробел внутри числа
    {"client": "c11", "amount": None},       # пусто
    {"client": "c05", "amount": "399.00"},
    {"client": "c31", "amount": "—"},        # прочерк
]

try:
    float(RAW[2]["amount"])
except ValueError as e:
    print(type(e).__name__)
    print(e)

Это **пятая из пяти ошибок вашей первой недели** — та самая, что была на лекции. Без `try` ячейка упала бы с трейсбеком, и последняя его строка была бы ровно такой:

```
ValueError: could not convert string to float: '1 200'
```

Никакой поломки Python здесь нет: строка `'1 200'` действительно не число.

Такой выгрузки в базе ПРАЙМ нет — там типы честные. Зато она будет в теме 10, когда данные придут из внешнего источника, и в любой первой рабочей задаче.

In [ ]:
def to_amount(value: str | None) -> float | None:
    """Превратить сумму из выгрузки в число. Вернуть None, если не вышло."""
    if value is None:
        return None
    text = str(value).replace("\u00a0", "").replace(" ", "").replace(",", ".")
    try:
        return float(text)
    except ValueError:
        return None


for row in RAW:
    print(f'{str(row["amount"]):>10}  ->  {to_amount(row["amount"])}')

Три вещи в этой функции стоит разобрать отдельно.

**Аннотации типов** `value: str | None` и `-> float | None` не проверяются при запуске и ничего не ускоряют. Они нужны человеку: из подписи видно, что на вход может прийти пустое значение и что на выходе тоже может не быть числа. Половина ошибок в аналитическом коде — это `None`, которого не ждали.

**`try/except ValueError`** ловит одну конкретную ошибку, а не все подряд. Писать `except:` без уточнения — плохая привычка: такой код проглотит и опечатку в имени переменной, и прерывание с клавиатуры.

**`\u00a0` — неразрывный пробел.** Его вставляют Excel и вёрстка; выглядит он как обычный пробел и не убирается обычным `.strip()`. Ошибка «а на глаз всё в порядке» родом отсюда.

In [ ]:
clean = [to_amount(row["amount"]) for row in RAW]
print("разобрано:", clean)
print("сумма    :", sum(v for v in clean if v is not None))
print("потеряно строк:", clean.count(None))

> **Считать надо и то, что не разобралось.** Строка «потеряно строк» здесь важнее суммы: молча выброшенные значения — это то, из-за чего отчёт расходится с бухгалтерией на месяц позже.

---

## 8. Словарь как справочник

In [ ]:
CASHBACK = {"Базовый": 0.01, "Расширенный": 0.03}

p = PAYMENTS[0]
CASHBACK[p["tariff"]] * p["amount"]

Словарь в роли справочника — самая частая его работа в аналитике. Это ровно то же, что соединение с таблицей `tariffs`, только маленькое и на месте.

In [ ]:
CASHBACK.get("Промо", 0.0)

`CASHBACK["Промо"]` упал бы с `KeyError` — третьей из пяти ошибок первой недели. `.get` возвращает умолчание.

Но осторожно: `.get` превращает громкую ошибку в тихий ноль. Иногда это то, что нужно, а иногда так и теряется целый тариф. **Решайте это осознанно, а не потому, что `.get` короче.**

---

## 9. Задачи со звёздочкой

Дальше — задачи на то же самое, но своими руками. **Они не сдаются и не оцениваются.** Смысл в другом: восемь задач ДЗ-1 будут ровно этих типов, и там они оцениваются. Прорешали здесь — дома это двадцать минут вместо двух часов.

Сколько успеете на паре — столько успеете, остальное доделайте дома. Проверка встроена: она знает правильный ответ, но не показывает его.

> Каждая задача — одна ячейка. Замените `None` на своё решение и запустите ячейку целиком.

In [ ]:
import hashlib


def _norm(x):
    if isinstance(x, bool):
        return repr(x)
    if isinstance(x, float):
        return f"{x:.2f}"
    if isinstance(x, int):
        return str(x)
    if isinstance(x, dict):
        return "{" + ",".join(f"{_norm(k)}:{_norm(v)}" for k, v in sorted(x.items(), key=lambda kv: _norm(kv[0]))) + "}"
    if isinstance(x, (set, frozenset)):
        return "{" + ",".join(sorted(_norm(v) for v in x)) + "}"
    if isinstance(x, (list, tuple)):
        return "[" + ",".join(_norm(v) for v in x) + "]"
    return repr(x)


ANSWERS = {
    "звёздочка 1": "0991f9b4ab",
    "звёздочка 2": "504735b37a",
    "звёздочка 3": "d6bbaacfc8",
    "звёздочка 4": "cdb019461e",
    "звёздочка 5": "4189bf8313",
    "звёздочка 6": "408fdb5156"
}


def check(task: str, got) -> None:
    """Сверить ответ с эталоном. Эталоны лежат хешами — подсмотреть нельзя."""
    if got is None:
        print(f"{task}: не решено")
    elif hashlib.sha256(_norm(got).encode()).hexdigest()[:10] == ANSWERS.get(task):
        print(f"{task}: верно")
    else:
        print(f"{task}: не сходится, получилось {got!r}")


print("проверка готова")

### ★ Задача 1. Средний успешный платёж

Сколько в среднем приносит один **успешный** платёж? Округлите до копеек — `round(x, 2)`.

In [ ]:
answer = None   # ← ваш расчёт

check("звёздочка 1", answer)

### ★ Задача 2. Подписки без денег

В `SUBSCRIBERS` — клиенты, у которых есть подписка. Найдите тех, у кого **нет ни одного успешного платежа**.

Ответ — множество (`set`) идентификаторов. Задача решается в одну строку, если вспомнить, что множества умеют вычитаться.

In [ ]:
SUBSCRIBERS = ["c17", "c04", "c23", "c11", "c05", "c31", "c42", "c08"]

answer = None   # ← ваш расчёт

check("звёздочка 2", answer)

### ★ Задача 3. Выручка по паре признаков

Посчитайте выручку по **паре** «тариф и канал» — то есть отдельно по каждому сочетанию.

Ответ — словарь, где ключ это строка вида `"Базовый|organic"`, а значение — сумма. Разделитель ровно такой: вертикальная черта без пробелов.

In [ ]:
answer = None   # ← ваш расчёт

check("звёздочка 3", answer)

### ★★ Задача 4. Свод по клиентам

Для каждого клиента, у которого есть успешные платежи, посчитайте **сколько их** и **на какую сумму**.

Ответ — словарь вида `{"c17": [2, 398.0], ...}`: сначала количество, потом сумма, списком.

In [ ]:
answer = None   # ← ваш расчёт

check("звёздочка 4", answer)

### ★★ Задача 5. Функция, которая принимает сколько угодно списков

Напишите `total(*groups)`, которая берёт **любое число** списков платежей и возвращает общую выручку по успешным. Звёздочка в подписи функции означает «сложи все переданные аргументы в кортеж».

Проверьте её, разрезав `PAYMENTS` на две части: `total(PAYMENTS[:7], PAYMENTS[7:])`.

In [ ]:
def total(*groups: list[dict]) -> float:
    ...   # ← ваш код


answer = None   # ← вызовите total на двух половинах PAYMENTS

check("звёздочка 5", answer)

### ★★★ Задача 6. Грязная выгрузка целиком

Возьмите `RAW` из раздела 7 и посчитайте сумму по всем строкам, которые удалось разобрать. Строки, где сумма не читается, в результат не идут — но и падать функция не должна.

Своей `to_amount` пользоваться можно.

In [ ]:
answer = None   # ← ваш расчёт

check("звёздочка 6", answer)

---

## 10. Что дальше

**ДЗ-1 выдано 13.09, дедлайн — 27.09, 23:59.** Срок общий для обеих групп. Приём со снижением балла — до 04.10. Текст задания — в папке [`задания/дз-1-окружение-и-репозиторий/`](../../задания/дз-1-окружение-и-репозиторий/) репозитория курса.

Что в нём: восемь задач на коллекции и функции — ровно тех типов, что в этом ноутбуке, — и ответы словами на вопросы по вашим данным. Сдаётся открытым pull request. Критерии и разбалловка — в тексте задания.

### Что унести с сегодняшнего дня

* Ветка и pull request — не бюрократия, а способ обсуждать работу построчно.
* Конфликт — это не авария. Git просто отказался решать за вас.
* `b = a` не копирует. Копирует `list(a)`.
* Генератор одноразовый. Второй `sum` по нему даст ноль и ничего не скажет.
* Считайте не только то, что разобралось, но и то, что потерялось.
* Пустая ячейка `except:` проглотит и вашу опечатку тоже.